# Medical Insurance Cost Prediction
## CSC-44112 Part 2 — Data Science Technical Report
**Task:** Predict individual medical insurance charges using regression models

**Dataset:** US Medical Insurance Costs (Kaggle — mirichoi0218/insurance)

---
## Setup — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
print('All libraries loaded successfully!')

---
## Section 3: Exploratory Data Analysis (EDA)
### 3.1 Load Dataset

In [ ]:
df = pd.read_csv('insurance.csv')
print('=== First 5 rows ===')
df.head()

### 3.2 Dataset Description & Descriptive Statistics

In [ ]:
print(f'Shape: {df.shape}')
print(f'\nData Types:\n{df.dtypes}')
print(f'\nDescriptive Statistics:')
df.describe()

### 3.3 Missing Data Analysis

In [ ]:
print('=== Missing Values ===')
print(df.isnull().sum())
print(f'\nTotal missing values: {df.isnull().sum().sum()}')
print('\nNo missing values — dataset is clean and ready for modelling.')

### 3.4 Distribution of Target Variable (Charges)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['charges'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Insurance Charges')
axes[0].set_xlabel('Charges ($)')
axes[0].set_ylabel('Frequency')

axes[1].hist(np.log(df['charges']), bins=50, color='teal', edgecolor='white')
axes[1].set_title('Log-Transformed Charges (more normal)')
axes[1].set_xlabel('Log(Charges)')
axes[1].set_ylabel('Frequency')

plt.suptitle('Figure 1: Charges Distribution (raw vs log-transformed)', y=1.02)
plt.tight_layout()
plt.savefig('fig1_charges_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.5 Charges by Smoker Status

In [ ]:
plt.figure(figsize=(8, 5))
df.boxplot(column='charges', by='smoker', patch_artist=True)
plt.title('Figure 2: Insurance Charges by Smoker Status')
plt.suptitle('')
plt.xlabel('Smoker')
plt.ylabel('Charges ($)')
plt.savefig('fig2_charges_by_smoker.png', dpi=150, bbox_inches='tight')
plt.show()

print('=== Mean Charges by Smoker ===')
print(df.groupby('smoker')['charges'].mean().round(2))

### 3.6 Scatter Plots — Age & BMI vs Charges

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = df['smoker'].map({'yes': 'red', 'no': 'steelblue'})

axes[0].scatter(df['age'], df['charges'], c=colors, alpha=0.5, s=20)
axes[0].set_title('Age vs Charges')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Charges ($)')

axes[1].scatter(df['bmi'], df['charges'], c=colors, alpha=0.5, s=20)
axes[1].set_title('BMI vs Charges')
axes[1].set_xlabel('BMI')
axes[1].set_ylabel('Charges ($)')

plt.suptitle('Figure 3: Scatter Plots (red = smoker, blue = non-smoker)', y=1.02)
plt.tight_layout()
plt.savefig('fig3_scatter_plots.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.7 Correlation Heatmap

In [ ]:
df_encoded = df.copy()
df_encoded['smoker'] = (df_encoded['smoker'] == 'yes').astype(int)
df_encoded['sex'] = (df_encoded['sex'] == 'male').astype(int)
df_encoded = pd.get_dummies(df_encoded, columns=['region'], drop_first=True)

plt.figure(figsize=(10, 7))
sns.heatmap(df_encoded.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Figure 4: Correlation Heatmap')
plt.tight_layout()
plt.savefig('fig4_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.8 Categorical Feature Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

df['sex'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'])
axes[0].set_title('Sex Distribution')
axes[0].set_xlabel('')

df['smoker'].value_counts().plot(kind='bar', ax=axes[1], color=['teal', 'salmon'])
axes[1].set_title('Smoker Distribution')
axes[1].set_xlabel('')

df['region'].value_counts().plot(kind='bar', ax=axes[2], color='steelblue')
axes[2].set_title('Region Distribution')
axes[2].set_xlabel('')

plt.suptitle('Figure 5: Categorical Feature Distributions', y=1.02)
plt.tight_layout()
plt.savefig('fig5_categorical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 4: Methodology
### 4.1 Preprocessing & Feature Engineering

In [ ]:
df_model = df.copy()

# Encode categorical variables
df_model['sex'] = LabelEncoder().fit_transform(df_model['sex'])
df_model['smoker'] = LabelEncoder().fit_transform(df_model['smoker'])
df_model = pd.get_dummies(df_model, columns=['region'], drop_first=True)

X = df_model.drop('charges', axis=1)
y = df_model['charges']

print(f'Features: {list(X.columns)}')

# 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}')

### 4.2 Model 1 — Linear Regression (Baseline)

In [ ]:
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])
lr_pipe.fit(X_train, y_train)
lr_pred = lr_pipe.predict(X_test)

print('=== Linear Regression Results ===')
print(f'R²:   {r2_score(y_test, lr_pred):.4f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, lr_pred)):.2f}')
print(f'MAE:  {mean_absolute_error(y_test, lr_pred):.2f}')

### 4.3 Model 2 — Random Forest

In [ ]:
rf_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42))
])
rf_pipe.fit(X_train, y_train)
rf_pred = rf_pipe.predict(X_test)

print('=== Random Forest Results ===')
print(f'R²:   {r2_score(y_test, rf_pred):.4f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, rf_pred)):.2f}')
print(f'MAE:  {mean_absolute_error(y_test, rf_pred):.2f}')

### 4.4 Model 3 — XGBoost with Hyperparameter Tuning

In [ ]:
param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [3, 5, 7],
    'model__learning_rate': [0.05, 0.1, 0.2],
}

xgb_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', xgb.XGBRegressor(random_state=42, verbosity=0))
])

grid_search = GridSearchCV(xgb_pipe, param_grid, cv=5, scoring='r2', n_jobs=-1)
grid_search.fit(X_train, y_train)

best_xgb = grid_search.best_estimator_
xgb_pred = best_xgb.predict(X_test)

print('=== XGBoost (Tuned) Results ===')
print(f'Best params: {grid_search.best_params_}')
print(f'R²:   {r2_score(y_test, xgb_pred):.4f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, xgb_pred)):.2f}')
print(f'MAE:  {mean_absolute_error(y_test, xgb_pred):.2f}')

---
## Section 5: Results and Evaluation
### 5.1 Model Comparison Table

In [ ]:
results = {
    'Model': ['Linear Regression', 'Random Forest', 'XGBoost (Tuned)'],
    'R²': [
        round(r2_score(y_test, lr_pred), 4),
        round(r2_score(y_test, rf_pred), 4),
        round(r2_score(y_test, xgb_pred), 4),
    ],
    'RMSE ($)': [
        round(np.sqrt(mean_squared_error(y_test, lr_pred)), 2),
        round(np.sqrt(mean_squared_error(y_test, rf_pred)), 2),
        round(np.sqrt(mean_squared_error(y_test, xgb_pred)), 2),
    ],
    'MAE ($)': [
        round(mean_absolute_error(y_test, lr_pred), 2),
        round(mean_absolute_error(y_test, rf_pred), 2),
        round(mean_absolute_error(y_test, xgb_pred), 2),
    ]
}
results_df = pd.DataFrame(results)
print('=== Model Comparison ===')
results_df

### 5.2 Predicted vs Actual Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
models_list = [('Linear Regression', lr_pred), ('Random Forest', rf_pred), ('XGBoost', xgb_pred)]

for ax, (name, pred) in zip(axes, models_list):
    ax.scatter(y_test, pred, alpha=0.4, s=15, color='steelblue')
    ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=1.5)
    ax.set_title(f'{name}\nR²={r2_score(y_test, pred):.3f}')
    ax.set_xlabel('Actual Charges ($)')
    ax.set_ylabel('Predicted Charges ($)')

plt.suptitle('Figure 6: Predicted vs Actual Charges', y=1.02)
plt.tight_layout()
plt.savefig('fig6_predicted_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.3 Residual Plot (XGBoost)

In [ ]:
residuals = y_test - xgb_pred
plt.figure(figsize=(8, 5))
plt.scatter(xgb_pred, residuals, alpha=0.4, s=15, color='teal')
plt.axhline(0, color='red', linestyle='--', lw=1.5)
plt.title('Figure 7: Residual Plot — XGBoost')
plt.xlabel('Predicted Charges ($)')
plt.ylabel('Residuals')
plt.savefig('fig7_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.4 Feature Importance (XGBoost)

In [ ]:
feature_importance = best_xgb.named_steps['model'].feature_importances_
feat_names = X.columns

plt.figure(figsize=(9, 5))
sorted_idx = np.argsort(feature_importance)
plt.barh(feat_names[sorted_idx], feature_importance[sorted_idx], color='steelblue')
plt.title('Figure 8: Feature Importance — XGBoost')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('fig8_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.5 Cross-Validation Score

In [ ]:
cv_scores = cross_val_score(best_xgb, X, y, cv=5, scoring='r2')
print('=== 5-Fold Cross-Validation (XGBoost) ===')
print(f'CV R² scores: {cv_scores.round(4)}')
print(f'Mean R²: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print('\nAll done! Check your folder for saved figures (fig1 to fig8).')